In [ ]:
# GEREKLI KUTUPHANELERIN EKLENMESI

import pandas as pd
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import BertTokenizer, TFBertForSequenceClassification
import tensorflow as tf
import torch

In [ ]:

# CSV DOSYASININ OKUNMASI

dframe = pd.read_csv("/content/vodafone_data_updated.csv", encoding='utf_8', on_bad_lines = 'skip', sep = ';')
dframe
# TITLE VE LINK SUTUNLARININ TABLODAN CIKARILMASI

#dframe = dframe.iloc[300:]
#dframe = dframe.drop(columns="Title",axis = 1)
#dframe = dframe.drop(columns="Link",axis=1)
#dframe = dframe.sample(frac = 1)
dframe
# TABLODAKI DUYGULARIN SAYISI

print("0: Olumsuz", "1: Olumlu", "2: Nötr\n")
dframe["Target"].value_counts()


In [ ]:
# METINDEN NOKTALAMA ISARETLERINI, RAKAMLARI CIKARMA
def clean_text(text):
    text = " ".join(str(text).split())
    text = text.lower()
    text = text.replace("\\n", " ")
    text = re.sub("[0-9]+", "", text)
    text = re.sub("%|(|)|-", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return "".join(text)

dframe["Clean"] = dframe.apply(lambda row: clean_text(row["Explanation"]), axis=1)
dframe

In [ ]:
# SATIRLARI NUMPY DIZISI YAPMA

X = dframe["Clean"]
y = dframe["Target"]

In [ ]:
# TEST VE EGITIM VERILERININ AYRILMASI
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23)
X_train_data, X_val, y_train_data, y_val = train_test_split(X_train, y_train, test_size=0.5)

In [ ]:

# BERT TOKENIZER EKLEME
tokenizer = BertTokenizer.from_pretrained('dbmdz/bert-base-turkish-cased')

In [ ]:
# BERT TOKENIZER UYGULAMA
max_len= 1024

X_train_encoded = tokenizer.batch_encode_plus(X_train_data.tolist(),
                                              padding=True,
                                              truncation=True,
                                              max_length = max_len,
                                              return_tensors='tf')

X_val_encoded = tokenizer.batch_encode_plus(X_val.tolist(),
                                              padding=True,
                                              truncation=True,
                                              max_length = max_len,
                                              return_tensors='tf')

X_test_encoded = tokenizer.batch_encode_plus(X_test.tolist(),
                                              padding=True,
                                              truncation=True,
                                              max_length = max_len,
                                              return_tensors='tf')

In [ ]:
# SINIFLANDIRMA MODELINI YUKLEME
model = TFBertForSequenceClassification.from_pretrained('dbmdz/bert-base-turkish-cased', num_labels = 3)

# MODELI COMPILE ETME
optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metric = tf.keras.metrics.SparseCategoricalAccuracy('accuracy')
model.compile(optimizer=optimizer, loss=loss, metrics=[metric])



In [ ]:
# MODELI EGITME
history = model.fit(
    [X_train_encoded['input_ids'], X_train_encoded['token_type_ids'], X_train_encoded['attention_mask']],
    y_train_data,
    validation_data=(
      [X_val_encoded['input_ids'], X_val_encoded['token_type_ids'], X_val_encoded['attention_mask']],y_val),
    batch_size=16,
    epochs=5
)

In [ ]:
# EVALUATE MODEL
test_loss, test_accuracy = model.evaluate(
    [X_test_encoded['input_ids'], X_test_encoded['token_type_ids'], X_test_encoded['attention_mask']],
    y_test
)
print(f'Test loss: {test_loss}, Test accuracy: {test_accuracy}')
# MODELI KAYDETME

path = 'path-to-save'
# Save tokenizer
tokenizer.save_pretrained(path +'/Tokenizer')

# Save model
model.save_pretrained(path +'/Model')

In [ ]:
# Load tokenizer
bert_tokenizer = BertTokenizer.from_pretrained(path +'/Tokenizer')

# Load model
bert_model = TFBertForSequenceClassification.from_pretrained(path +'/Model')

In [33]:
### DENEME
pred = bert_model.predict(
	[X_test_encoded['input_ids'], X_test_encoded['token_type_ids'], X_test_encoded['attention_mask']])

# pred is of type TFSequenceClassifierOutput
logits = pred.logits

# Use argmax along the appropriate axis to get the predicted labels
pred_labels = tf.argmax(logits, axis=1)

# Convert the predicted labels to a NumPy array
pred_labels = pred_labels.numpy()

label = {
	0: 'Olumsuz',
	1: 'Olumlu',
  2: 'Nötr'
}

# Map the predicted labels to their corresponding strings using the label dictionary
pred_labels = [label[i] for i in pred_labels]
Actual = [label[i] for i in y_test]

print('Predicted Label :', pred_labels[:10])
print('Actual Label :', Actual[:10])

print("Classification Report: \n", classification_report(Actual, pred_labels))
# KULLANICI GIRISI ILE TAHMIN FONKSIYONU

def Get_sentiment(Review, Tokenizer=bert_tokenizer, Model=bert_model):
	# Convert Review to a list if it's not already a list
	if not isinstance(Review, list):
		Review = [Review]

	Input_ids, Token_type_ids, Attention_mask = Tokenizer.batch_encode_plus(Review,
																			padding=True,
																			truncation=True,
																			max_length=1024,
																			return_tensors='tf').values()
	prediction = Model.predict([Input_ids, Token_type_ids, Attention_mask])

	# Use argmax along the appropriate axis to get the predicted labels
	pred_labels = tf.argmax(prediction.logits, axis = 1)

	# Convert the TensorFlow tensor to a NumPy array and then to a list to get the predicted sentiment labels
	pred_labels = [label[i] for i in pred_labels.numpy().tolist()]
	return pred_labels

# KULLANICI GIRISI ILE TAHMIN FONKSIYONU
Review = "ne iyi ne de kotu. Kararsizim"
Get_sentiment(Review)

0: Olumsuz 1: Olumlu 2: Nötr



tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/251k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/5
121/121 [==============================] - 81s 292ms/step - loss: 0.7044 - accuracy: 0.7060 - val_loss: 0.4347 - val_accuracy: 0.8329
Epoch 2/5
121/121 [==============================] - 31s 257ms/step - loss: 0.3487 - accuracy: 0.8711 - val_loss: 0.5066 - val_accuracy: 0.8153
Epoch 3/5
121/121 [==============================] - 31s 253ms/step - loss: 0.1769 - accuracy: 0.9400 - val_loss: 0.5088 - val_accuracy: 0.8257
Epoch 4/5
121/121 [==============================] - 30s 248ms/step - loss: 0.1048 - accuracy: 0.9664 - val_loss: 0.5527 - val_accuracy: 0.8526
Epoch 5/5
31/31 [==============================] - 6s 80ms/step - loss: 0.7810 - accuracy: 0.8211
Test loss: 0.7809654474258423, Test accuracy: 0.8210961818695068


Some layers from the model checkpoint at path-to-save/Model were not used when initializing TFBertForSequenceClassification: ['dropout_37']
- This IS expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at path-to-save/Model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForSequenceClassification for predictions without further training.


31/31 [==============================] - 6s 79ms/step
Predicted Label : ['Olumlu', 'Olumsuz', 'Nötr', 'Olumsuz', 'Olumsuz', 'Olumsuz', 'Nötr', 'Olumsuz', 'Nötr', 'Olumlu']
Actual Label : ['Olumlu', 'Olumsuz', 'Olumlu', 'Nötr', 'Olumsuz', 'Olumsuz', 'Nötr', 'Olumsuz', 'Nötr', 'Olumlu']
Classification Report: 
               precision    recall  f1-score   support

        Nötr       0.53      0.61      0.57       129
      Olumlu       0.82      0.75      0.78       243
     Olumsuz       0.89      0.90      0.89       595

    accuracy                           0.82       967
   macro avg       0.75      0.75      0.75       967
weighted avg       0.83      0.82      0.82       967

1/1 [==============================] - 0s 44ms/step


['Nötr']

In [41]:
Review = "diş ağrısı çekiyorum"
Get_sentiment(Review)

1/1 [==============================] - 0s 41ms/step


['Olumsuz']

In [45]:
Review = "acı sızı"
Get_sentiment(Review)


1/1 [==============================] - 0s 44ms/step


['Olumsuz']

In [46]:
Review = "Akşam yemeğini ailemle birlikte yiyeceğim."
Get_sentiment(Review)

1/1 [==============================] - 0s 46ms/step


['Olumlu']

In [52]:
Review = "ama"
Get_sentiment(Review)

1/1 [==============================] - 0s 43ms/step


['Nötr']

In [53]:
!zip -r model_and_tokenizer.zip path-to-save


  adding: path-to-save/ (stored 0%)
  adding: path-to-save/Tokenizer/ (stored 0%)
  adding: path-to-save/Tokenizer/vocab.txt (deflated 53%)
  adding: path-to-save/Tokenizer/tokenizer_config.json (deflated 75%)
  adding: path-to-save/Tokenizer/special_tokens_map.json (deflated 42%)
  adding: path-to-save/Model/ (stored 0%)
  adding: path-to-save/Model/config.json (deflated 51%)
  adding: path-to-save/Model/tf_model.h5 (deflated 7%)


In [54]:
from google.colab import files
files.download("model_and_tokenizer.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>